In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp
import sys
sys.path.insert(0, '/cvmfs/larsoft.opensciencegrid.org/spack-fnal-v1.0.0/opt/spack/linux-x86_64_v2/root-6.28.12-vgs6mjswsg36hl3oarsrsyc2dcua6khe/lib/root')

import ROOT

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import gc

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from analysis_village.cc1pi.var_configs import *



from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

from analysis_village.cc1pi.Optimize import OptimizationUtils
from analysis_village.cc1pi.Optimize import ConfusionMatricesUtils

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
"""
kfold_importance_check.py

Comprueba si la importancia de las variables de entrada (en particular la
que domina, p.ej. chi2_pol0/chi2_exp) es ESTABLE a través de distintos
splits de los datos, o si varía mucho de un fold a otro -- lo cual
apuntaría a que el efecto es ruido/varianza por tener pocas estadisticas
de fondo, y no un efecto físico consistente.

Usa:
  - StratifiedKFold para mantener la proporción senal/fondo en cada fold
  - pesos balanceados recalculados EN CADA FOLD usando solo el train de
    ese fold (para no filtrar información del fold de validación)
  - permutation_importance sobre el fold de VALIDACION (no en train),
    para evitar el sesgo de la importancia por impureza (Gini) de los
    arboles, que favorece a variables continuas con muchos puntos de
    corte posibles.

Al final, imprime y grafica la media +/- std de la importancia de cada
variable a través de los folds, y el ranking de importancia en cada fold
individual, para que puedas ver si la variable dominante lo es siempre
o solo en algunos splits.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance


def run_kfold_importance_check(
    signal_df,
    bkg_df,
    input_cols,
    model,
    n_splits=5,
    random_state=40,
    n_repeats=10,
    scoring="roc_auc",
):
    """
    Parameters
    ----------
    signal_df, bkg_df : DataFrames con las columnas en input_cols
    input_cols : list[str] nombres de las variables de entrada
    model : un estimador/pipeline sin entrenar (p.ej. model_vec["BDTG"]);
            se clona en cada fold, así que puedes pasar el mismo objeto
            que usas en tu train_all_models.
    n_splits : numero de folds (5 es un buen default; con solo ~6800
               eventos de fondo, no bajes de 5 o los folds de validación
               se quedan con muy pocos eventos de fondo)
    n_repeats : repeticiones de la permutación por fold (permutation_importance
                ya promedia internamente, pero mas repeats = menos ruido)
    scoring : métrica usada para permutation_importance

    Returns
    -------
    importances_df : DataFrame (n_splits x n_features) con la importancia
                      de cada variable en cada fold
    summary_df : DataFrame con mean, std, y coeficiente de variacion (CV)
                 por variable, ordenado por importancia media descendente
    """
    # 1. Construir X, y combinando senal y fondo
    X_sig = signal_df[input_cols].values
    X_bkg = bkg_df[input_cols].values
    y_sig = np.ones(len(X_sig))
    y_bkg = np.zeros(len(X_bkg))

    X = np.vstack([X_sig, X_bkg])
    y = np.hstack([y_sig, y_bkg])

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_importances = []
    fold_aucs = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # 2. Pesos balanceados calculados SOLO con el train de este fold
        n_train_sig = int((y_train == 1).sum())
        n_train_bkg = int((y_train == 0).sum())
        weight_bkg = n_train_sig / n_train_bkg
        sample_weights = np.where(y_train == 1, 1.0, weight_bkg)

        # 3. Entrenar
        fold_model = clone(model)
        fold_model.fit(X_train, y_train, classifier__sample_weight=sample_weights)

        # 4. Permutation importance en el fold de VALIDACION
        result = permutation_importance(
            fold_model,
            X_val,
            y_val,
            scoring=scoring,
            n_repeats=n_repeats,
            random_state=random_state,
            n_jobs=-1,
        )
        fold_importances.append(result.importances_mean)

        # AUC del fold, util para ver si el modelo generaliza bien en general
        from sklearn.metrics import roc_auc_score

        y_score = (
            fold_model.predict_proba(X_val)[:, 1]
            if hasattr(fold_model, "predict_proba")
            else fold_model.decision_function(X_val)
        )
        fold_auc = roc_auc_score(y_val, y_score)
        fold_aucs.append(fold_auc)

        ranking = np.argsort(result.importances_mean)[::-1]
        ranked_names = [input_cols[i] for i in ranking]
        print(f"Fold {fold_idx}: AUC={fold_auc:.4f} | ranking (mayor a menor): {ranked_names}")

    importances_df = pd.DataFrame(fold_importances, columns=input_cols)
    importances_df.index = [f"fold_{i+1}" for i in range(n_splits)]

    summary_df = pd.DataFrame(
        {
            "mean_importance": importances_df.mean(axis=0),
            "std_importance": importances_df.std(axis=0),
        }
    )
    # coeficiente de variacion: std/mean. Alto (>0.5-1) => inestable entre folds
    summary_df["cv"] = summary_df["std_importance"] / summary_df["mean_importance"].abs()
    summary_df = summary_df.sort_values("mean_importance", ascending=False)

    print("\n=== Resumen (ordenado por importancia media) ===")
    print(summary_df.to_string(float_format=lambda v: f"{v:.4f}"))
    print(f"\nAUC por fold: {[f'{a:.4f}' for a in fold_aucs]}")
    print(f"AUC medio: {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}")

    return importances_df, summary_df, fold_aucs


def plot_importance_stability(summary_df, importances_df, title="Estabilidad de importancia entre folds"):
    """
    Grafica barras horizontales con la media de importancia y barras de
    error mostrando la std entre folds. Una variable "real" deberia tener
    una barra de error pequena relativa a su media; una variable inestable
    (artefacto de muestra pequena) tendra una barra de error grande, o
    incluso solapara con cero o con otras variables.
    """
    fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(summary_df))))

    order = summary_df.index[::-1]  # de menor a mayor para que el barh quede ordenado
    means = summary_df.loc[order, "mean_importance"]
    stds = summary_df.loc[order, "std_importance"]

    ax.barh(order, means, xerr=stds, color="skyblue", edgecolor="black", capsize=4)
    ax.set_xlabel("Permutation Importance (media +/- std entre folds)")
    ax.set_title(title)
    ax.axvline(0, color="gray", linewidth=0.8)
    plt.tight_layout()
    return fig




In [ ]:
## Check keys in each file
SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]

cols_to_keep = [
('nu_categ', '', '', ''),
('nu_categ_proton_reduced', '', '', ''),
('genie_categ', '', '', ''),
('genie_mode', '', '', ''),
]
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

pot_weight_col = ('slc', 'wgt', '', '', '', '')

optimization_file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_1e20_training_update_cv_BDT_train.df"
#development_sample_file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_pruned_for_data_mc_comp.df"
development_sample_file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/wrong_energy_cut/mc_ar23p_pruned_no_syst_for_BDT_comp.df"

In [ ]:
optimization_df = load_df(optimization_file, keys2load, 100, filter_df = True, reprocess_df = True, reprocess_truth = True)
optimization_evt_df = optimization_df['cc1pi']
optimization_hdr_df = optimization_df['hdr']
optimization_nu_df = optimization_df['nudf']

print("data_tot_pot: %.3e" %(data_tot_pot))

optimization_tot_pot = optimization_hdr_df['pot'].sum()
mc_pot_scale = data_tot_pot / optimization_tot_pot
print("mc_tot_pot: %.3e" %(optimization_tot_pot))
print("mc_pot_scale: %.3e" %(mc_pot_scale))
optimization_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(optimization_evt_df))

optimization_evt_df = perform_truth_matching(optimization_evt_df, optimization_nu_df[cols_to_keep])

evt_df = optimization_evt_df

In [ ]:
development_sample_df = load_df(development_sample_file, keys2load, 100, filter_df = False, reprocess_df = False, reprocess_truth = False)
dev_sample_evt_df = development_sample_df['cc1pi']
dev_sample_hdr_df = development_sample_df['hdr']
#dev_sample_nu_df = development_sample_df['nudf']

print("data_tot_pot: %.3e" %(data_tot_pot))

dev_sample_tot_pot = dev_sample_hdr_df['pot'].sum()
mc_pot_scale = data_tot_pot / dev_sample_tot_pot
print("mc_tot_pot: %.3e" %(dev_sample_tot_pot))
print("mc_pot_scale: %.3e" %(mc_pot_scale))
dev_sample_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(dev_sample_evt_df))

#dev_sample_evt_df = perform_truth_matching(dev_sample_evt_df, dev_sample_nu_df[cols_to_keep])  

# BDT Training Start

In [ ]:
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/BDTTrainingProton"
if not os.path.exists(save_fig_dir):
    os.makedirs(save_fig_dir)

In [ ]:
import BDTTrainingUtils

In [ ]:
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_chi2_exp_pol  = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_frac_50 = ('pfp', 'trk', 'frac50', '', '', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')
BDT_columns = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_frac_50]

cut_mask = evt_df.slc.cut.obvious_cosmic & CutMasks.t0_cut_mask(evt_df) & CutMasks.nu_score_cut_mask(evt_df) & CutMasks.is_inside_FV_cut_mask(evt_df)  & (evt_df.truth.nu_categ != "cosmic")

#Define the track df
signal_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 2212)  & (evt_df.pfp.trk.truth.p.end_process == 7 ) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for training
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]
signal_df = signal_df[signal_df.pfp.is_exiting == False]

#Only with good values
bdt_mask_signal = BDTTrainingUtils.bdt_quality_mask(signal_df, BDT_columns)
bdt_mask_bkg    = BDTTrainingUtils.bdt_quality_mask(bkg_df, BDT_columns)
signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]

#MIP mask
signal_df = signal_df[CutMasks.is_MIP_candidate_mask(signal_df)]
bkg_df    = bkg_df[CutMasks.is_MIP_candidate_mask(bkg_df)]

In [ ]:
BDT_columns = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_frac_50]
model_vec = BDTTrainingUtils.create_models_for_columns(BDT_columns)

In [ ]:
# Train all models
trained_models, X_train_normal, X_test_normal, y_train_normal, y_test_normal = BDTTrainingUtils.train_all_models(
    signal_df,
    bkg_df,
    BDT_columns,
    model_vec,
    test_size=0.3
)

In [ ]:
'''
model_vec = create_models_for_columns(BDT_columns)
importances_df, summary_df, fold_aucs = run_kfold_importance_check(
    signal_df, bkg_df, BDT_columns, model_vec["BDTG"], n_splits=5
)
fig = plot_importance_stability(summary_df, importances_df)
fig.savefig("/mnt/user-data/outputs/kfold_importance_stability.png", dpi=150)
'''

In [ ]:
xgb_scores_train, xgb_scores_test   = BDTTrainingUtils.get_scores(trained_models["XGB"], X_train_normal, X_test_normal)
bdt_scores_train, bdt_scores_test   = BDTTrainingUtils.get_scores(trained_models["BDT"], X_train_normal, X_test_normal)
bdtg_scores_train, bdtg_scores_test = BDTTrainingUtils.get_scores(trained_models["BDTG"], X_train_normal, X_test_normal)
bdtb_scores_train, bdtb_scores_test = BDTTrainingUtils.get_scores(trained_models["BDTB"], X_train_normal, X_test_normal)
random_forest_scores_train, random_forest_scores_test = BDTTrainingUtils.get_scores(trained_models["RF"], X_train_normal, X_test_normal)

In [ ]:
fig = plt.figure(figsize=(6,6))
#BDTTrainingUtils.plot_roc(y_test_normal, bdt_scores_test, "BDT")
BDTTrainingUtils.plot_roc(y_test_normal, bdtg_scores_test, "BDTG")
#BDTTrainingUtils.plot_roc(y_test_normal, bdtb_scores_test, "BDTB")
#BDTTrainingUtils.plot_roc(y_test_normal, bdtb_scores_test, "XGB")
#BDTTrainingUtils.plot_roc(y_test_normal, random_forest_scores_test, "RandomForest")
plt.plot([0,1],[0,1], "k--")
plt.xlabel("Background efficiency")
plt.ylabel("Signal efficiency")
plt.legend()

save_path = os.path.join(save_fig_dir, "ROC_curves.pdf")
fig.savefig(save_path, format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
sig_train_bdtg = bdtg_scores_train[y_train_normal==1]
sig_test_bdtg  = bdtg_scores_test[y_test_normal==1]
bkg_train_bdtg = bdtg_scores_train[y_train_normal==0]
bkg_test_bdtg  = bdtg_scores_test[y_test_normal==0]

sig_train_bdt = bdt_scores_train[y_train_normal==1]
sig_test_bdt  = bdt_scores_test[y_test_normal==1]
bkg_train_bdt = bdt_scores_train[y_train_normal==0]
bkg_test_bdt  = bdt_scores_test[y_test_normal==0]


sig_train_xgb = xgb_scores_train[y_train_normal==1]
sig_test_xgb = xgb_scores_test[y_test_normal==1]
bkg_train_xgb = xgb_scores_train[y_train_normal==0]
bkg_test_xgb = xgb_scores_test[y_test_normal==0]

In [ ]:

save_path = os.path.join(save_fig_dir, "Importance.pdf")
df_imp = BDTTrainingUtils.plot_importance_no_transform(trained_models["BDTG"], BDT_columns, name_map= BDTTrainingUtils.variables_name_map, save_path = save_path)


In [ ]:

save_path = os.path.join(save_fig_dir, "correlation_matrices.pdf")
corr_signal, corr_bkg = BDTTrainingUtils.plot_correlation_matrices(signal_df, bkg_df, BDT_columns, name_map = BDTTrainingUtils.variables_name_map, save_folder = save_path)

In [ ]:
#corr_matrix = BDTTrainingUtils.plot_correlation_matrix(signal_df, BDT_columns, title="Input variable correlations")
#corr_matrix = BDTTrainingUtils.plot_correlation_matrix(bkg_df, BDT_columns, title="Input variable correlations")

In [ ]:
# Plot
save_path = os.path.join(save_fig_dir, "training_test_BDT_response.pdf")
BDTTrainingUtils.plot_response(sig_train_bdt, sig_test_bdt, bkg_train_bdt, bkg_test_bdt, title = "BDT Response", signal_label=r"$\mu/\pi$", bkg_label="proton", save_path = save_path)
save_path = os.path.join(save_fig_dir, "training_test_BDTG_response.pdf")
BDTTrainingUtils.plot_response(sig_train_bdtg, sig_test_bdtg, bkg_train_bdtg, bkg_test_bdtg, title = "BDTG Response", signal_label=r"$\mu/\pi$", bkg_label="proton", save_path = save_path)
save_path = os.path.join(save_fig_dir, "training_test_XGB_response.pdf")
BDTTrainingUtils.plot_response(sig_train_xgb, sig_test_xgb, bkg_train_xgb, bkg_test_xgb, title ="XGB Response", signal_label=r"$\mu/\pi$", bkg_label="proton", save_path = save_path)

In [ ]:
with open("test_bdts/bdt_model_proton_v2.pkl", "wb") as f:
    pickle.dump(trained_models["BDT"], f)

with open("test_bdts/bdtg_model_proton.pkl", "wb") as f:
    pickle.dump(trained_models["BDTG"], f)
    
with open("test_bdts/xgb_model.pkl", "wb") as f:
    pickle.dump(trained_models["XGB"], f)

# BDT Tests and Optimization

In [ ]:
from analysis_village.cc1pi.Optimize import OptimizationUtils
from analysis_village.cc1pi.Optimize import ConfusionMatricesUtils
from analysis_village.cc1pi.makedf import make_cc1pidf


BDT_input_columns = [
    ('pfp','trk','chi2pid','best','chi2_muon',''),
    ('pfp','trk','chi2pid','best','chi2_proton',''),
    ('pfp','trk','chi2_exp_pol','','',''),
    ('pfp','trk','frac50','','','')
]

col_BDT_score_proton = ('pfp','BDT_score_proton','','','','')
col_BDTG_score_proton = ('pfp','BDTG_score_proton','','','','')
col_XGB_score_proton = ('pfp','XGB_score_proton','','','','')

'''
evt_df = make_cc1pidf.add_bdt_score(
    evt_df,
    "test_bdts/bdt_model_proton_v2.pkl",
    BDT_input_columns,
    col_BDT_score_proton
)
'''

evt_df = make_cc1pidf.add_bdt_score(
    evt_df,
    "test_bdts/bdtg_model_proton.pkl",
    #"bdtg_model_proton.pkl",
    BDT_input_columns,
    col_BDTG_score_proton
)
'''
evt_df = make_cc1pidf.add_bdt_score(
    evt_df,
    "test_bdts/xgb_model.pkl",
    BDT_input_columns,
    col_XGB_score_proton
)

dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "test_bdts/bdt_model_proton_v2.pkl",
    BDT_input_columns,
    col_BDT_score_proton
)
'''

'''
dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "bdtg_model_proton.pkl",
    BDT_input_columns,
    col_BDTG_score_proton
)
'''

'''
dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "test_bdts/xgb_model.pkl",
    BDT_input_columns,
    col_XGB_score_proton
)
'''

In [ ]:
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_chi2_exp_pol  = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_frac_50 = ('pfp', 'trk', 'frac50', '', '', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')

cut_mask = (evt_df.slc.cut.obvious_cosmic & evt_df.slc.cut.t0 & evt_df.slc.cut.inside_FV & (evt_df.slc.nu_score > CTE.min_nu_score) & evt_df.slc.cut.track
        & evt_df.slc.cut.MIP_candidates & evt_df.slc.cut.shower & evt_df.slc.cut.angle)

#Define the track df
signal_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 2212)  & (evt_df.pfp.trk.truth.p.end_process == 7 ) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for training
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]
signal_df = signal_df[signal_df.pfp.is_exiting == False]

#Only with good values
bdt_mask_signal = BDTTrainingUtils.bdt_quality_mask(signal_df, BDT_columns)
bdt_mask_bkg    = BDTTrainingUtils.bdt_quality_mask(bkg_df, BDT_columns)
signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]

#MIP mask
signal_df = signal_df[CutMasks.is_MIP_candidate_mask(signal_df)]
bkg_df    = bkg_df[CutMasks.is_MIP_candidate_mask(bkg_df)]

In [ ]:
'''
col_BDT_score_proton = ('pfp','BDT_score_proton','','','','')
col_BDTG_score_proton = ('pfp','BDTG_score_proton','','','','')
col_XGB_score_proton = ('pfp','XGB_score_proton','','','','')
col_BDT_TMVA_og_score_proton = ('pfp','trk','og_bdt_proton_score','','','')
col_BDT_TMVA_retrain_score_proton = ('pfp','trk','retrain_bdt_proton_score','','','')

sing = ">"
bdt_opt_len, best_bdt, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_BDT_score_proton,
    cut_type=sing,
    xlabel="length",
    title="BDT optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-0.6, 1.2),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0, # <-- new parameter
)

bdtg_opt_len, best_bdtg, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_BDTG_score_proton,
    cut_type=sing,
    xlabel="length",
    title="BDTG optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-4, 8),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0, # <-- new parameter
)

xgb_bdt_df, best_xgb, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_XGB_score_proton,
    cut_type=sing,
    xlabel="length",
    title="XGB optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(0, 1),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0, # <-- new parameter
)
'''

In [ ]:

col_BDT_score_proton = ('pfp','BDT_score_proton','','','','')
col_BDTG_score_proton = ('pfp','BDTG_score_proton','','','','')
col_XGB_score_proton = ('pfp','XGB_score_proton','','','','')
col_BDT_TMVA_og_score_proton = ('pfp','trk','og_bdt_proton_score','','','')
col_BDT_TMVA_retrain_score_proton = ('pfp','trk','retrain_bdt_proton_score','','','')

sing = ">"
'''
bdt_opt_len, best_bdt, fig = OptimizationUtils.optimize_cut_accuracy(
    signal_df,
    bkg_df,
    column= col_BDT_score_proton,
    cut_type=sing,
    xlabel="length",
    title="BDT optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-0.6, 1.2),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False
)
'''


save_path = os.path.join(save_fig_dir, "BDTG_score_accuracy_optimization_old_fv.pdf")
bdtg_opt_len, best_bdtg, fig = OptimizationUtils.optimize_cut_accuracy(
    signal_df,
    bkg_df,
    column= col_BDTG_score_proton,
    cut_type=sing,
    xlabel="BDTG Score",
    title="",
    save_folder = save_path,
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-5, 6),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False
)

'''
xgb_bdt_df, best_xgb, fig = OptimizationUtils.optimize_cut_accuracy(
    signal_df,
    bkg_df,
    column= col_XGB_score_proton,
    cut_type=sing,
    xlabel="length",
    title="XGB optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(0, 1),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False
)
'''

In [ ]:
cut_mask = (evt_df.slc.cut.obvious_cosmic & evt_df.slc.cut.t0 & CutMasks.InFV_strict(evt_df) & (evt_df.slc.nu_score > CTE.min_nu_score) & evt_df.slc.cut.track
        & evt_df.slc.cut.MIP_candidates & evt_df.slc.cut.shower & evt_df.slc.cut.angle)

#Define the track df
signal_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 2212)  & (evt_df.pfp.trk.truth.p.end_process == 7 ) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for training
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]
signal_df = signal_df[signal_df.pfp.is_exiting == False]

#Only with good values
bdt_mask_signal = BDTTrainingUtils.bdt_quality_mask(signal_df, BDT_columns)
bdt_mask_bkg    = BDTTrainingUtils.bdt_quality_mask(bkg_df, BDT_columns)
signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]

#MIP mask
signal_df = signal_df[CutMasks.is_MIP_candidate_mask(signal_df)]
bkg_df    = bkg_df[CutMasks.is_MIP_candidate_mask(bkg_df)]




save_path = os.path.join(save_fig_dir, "BDTG_score_accuracy_optimization_new_fv.pdf")
bdtg_opt_len, best_bdtg, fig = OptimizationUtils.optimize_cut_accuracy(
    signal_df,
    bkg_df,
    column= col_BDTG_score_proton,
    cut_type=sing,
    xlabel="BDTG Score",
    title="",
    save_folder = save_path,
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-4, 8.5),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False
)


In [ ]:
SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]
def proton_test_BDT_cut_mask(df, group_levels, column, cut):
    BDT_proton_df = df[(CutMasks.is_MIP_candidate_mask(df)) & (df[column] > cut)]
    
    # Count how many pfps per slice
    candidate_counts = BDT_proton_df.groupby(level=group_levels).size()
 
    # Get only slices with at least 2 pfps
    valid_slices = candidate_counts[candidate_counts == 2].index

    # Apply the mask to original DataFrame
    final_mask = pd.Series(df.index.droplevel('rec.slc.reco.pfp..index').isin(valid_slices), index=df.index)

    return final_mask

In [ ]:
'''
cut_mask = (evt_df.slc.cut.obvious_cosmic & CutMasks.t0_cut_mask(evt_df) & CutMasks.nu_score_cut_mask(evt_df) & CutMasks.is_inside_FV_cut_mask(evt_df) & CutMasks.track_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(evt_df, SLICE_LEVELS) 
        & CutMasks.chi2_cut_mask(evt_df,SLICE_LEVELS))
'''

cut_mask = (dev_sample_evt_df.slc.cut.obvious_cosmic & dev_sample_evt_df.slc.cut.t0 & dev_sample_evt_df.slc.cut.inside_FV & (dev_sample_evt_df.slc.nu_score > CTE.min_nu_score) & dev_sample_evt_df.slc.cut.track
        & dev_sample_evt_df.slc.cut.MIP_candidates & dev_sample_evt_df.slc.cut.shower & dev_sample_evt_df.slc.cut.angle)

print("control")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask], target_categ="CC1pi")
'''
print("TMVA og")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask  & proton_test_BDT_cut_mask(dev_sample_evt_df, SLICE_LEVELS, col_BDT_TMVA_og_score_proton, best_TMVA_og['cut'])], target_categ="CC1pi")
print("TMVA retrain")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask  & proton_test_BDT_cut_mask(dev_sample_evt_df, SLICE_LEVELS, col_BDT_TMVA_retrain_score_proton, best_TMVA_retrain['cut'])], target_categ="CC1pi")
print("BDT")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask & proton_test_BDT_cut_mask(dev_sample_evt_df, SLICE_LEVELS, col_BDT_score_proton, best_bdt['cut'])], target_categ="CC1pi")
'''

print("BDTG")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask & proton_test_BDT_cut_mask(dev_sample_evt_df, SLICE_LEVELS, ('pfp', 'trk', 'bdt_proton_score', '', '', ''), best_bdtg['cut'])], target_categ="CC1pi")

'''
print("BDTG 066")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask & proton_test_BDT_cut_mask(dev_sample_evt_df, SLICE_LEVELS, col_BDTG_score_proton, 0.62)], target_categ="CC1pi")
print("XGB")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask & proton_test_BDT_cut_mask(dev_sample_evt_df, SLICE_LEVELS, col_XGB_score_proton, best_xgb['cut'])], target_categ="CC1pi")
'''

In [ ]:
def build_pid_confusion_matrix(df, bdtg_threshold=0.5, key=('pfp', 'BDTG_score_proton', '', '', '', ''), weight_key=None):
    """
    Builds a confusion matrix for Proton vs Muon/Pion identification.
    
    True Labels: [Muon/Pion, Proton]
    Reco Labels: [Reco Muon/Pion, Reco Proton]
    """
    import numpy as np
    import pandas as pd

    # 1. Define Keys
    pdg_key = ('pfp', 'trk', 'truth', 'p', 'pdg', '')
    
    # 2. Filter for specific particles only (13, 211, 2212)
    # We use .copy() to avoid SettingWithCopy warnings
    mask_valid = df[pdg_key].abs().isin([13, 211, 2212])
    df_filtered = df[mask_valid].copy()
    
    if df_filtered.empty:
        print("Warning: No valid PDGs (13, 211, 2212) found in DataFrame.")
        return np.zeros((2,2)), [], []

    # 3. Assign True Category (0: Muon/Pion, 1: Proton)
    df_filtered['true_pid'] = np.where(df_filtered[pdg_key].abs() == 2212, 1, 0)
    
    # 4. Assign Reco Category (1: Proton if < threshold, 0: Muon/Pion)
    # Logic: Score < threshold -> Reco Proton
    df_filtered['reco_pid'] = np.where(df_filtered[key] < bdtg_threshold, 1, 0)
    
    # 5. Define Weights
    # If no weight is provided, count each row as 1.0
    if weight_key is None:
        weights = np.ones(len(df_filtered))
    else:
        weights = df_filtered[weight_key]

    # 6. Build Matrix using pd.crosstab (much safer than groupby.sum for CMs)
    # This creates the matrix directly with reco on index and true on columns
    cm_df = pd.crosstab(
        df_filtered['reco_pid'], 
        df_filtered['true_pid'], 
        values=weights, 
        aggfunc='sum'
    ).fillna(0)

    # Ensure the matrix is 2x2 even if a category is missing
    cm = np.zeros((2, 2))
    for r in [0, 1]:
        for t in [0, 1]:
            if r in cm_df.index and t in cm_df.columns:
                cm[r, t] = cm_df.loc[r, t]

    # Labels
    x_labels = [r"True $\mu/\pi$", "True Proton"]
    y_labels = [r"Reco $\mu/\pi$", "Reco Proton"]
    
    return cm, x_labels, y_labels

In [ ]:

from analysis_village.cc1pi.Optimize import ConfusionMatricesUtils

cut_mask =  (evt_df.slc.cut.obvious_cosmic & evt_df.slc.cut.t0 & (evt_df.slc.cut.inside_FV == True) & (evt_df.slc.nu_score > CTE.min_nu_score) & evt_df.slc.cut.track
        & evt_df.slc.cut.MIP_candidates & evt_df.slc.cut.shower & evt_df.slc.cut.angle)

cm_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)|(abs(evt_df.pfp.trk.truth.p.pdg) == 2212)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for training
cm_df = cm_df[cm_df.pfp.is_exiting == False]
cm_df = cm_df[CutMasks.is_MIP_candidate_mask(cm_df)]
cm, x_labs, y_labs = build_pid_confusion_matrix(cm_df, bdtg_threshold = best_bdtg['cut'], key = ('pfp', 'BDTG_score_proton', '', '', '', ''))
cm_plot = ConfusionMatricesUtils.plot_confusion_matrix(cm, x_labs, y_labs)

save_path = os.path.join(save_fig_dir, "BDT_proton_confusion_matrix_1e20_training_sample.pdf")
cm_plot.savefig(save_path, bbox_inches='tight', dpi=dpi)
print(f"Saved: {save_path}")
plt.show()



cut_mask =  (dev_sample_evt_df.slc.cut.obvious_cosmic & dev_sample_evt_df.slc.cut.t0 & (dev_sample_evt_df.slc.cut.inside_FV == True) & (dev_sample_evt_df.slc.nu_score > CTE.min_nu_score) & dev_sample_evt_df.slc.cut.track
        & dev_sample_evt_df.slc.cut.MIP_candidates & dev_sample_evt_df.slc.cut.shower & dev_sample_evt_df.slc.cut.angle)

cm_df = dev_sample_evt_df[
    cut_mask &
    ((abs(dev_sample_evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(dev_sample_evt_df.pfp.trk.truth.p.pdg) == 13)|(abs(dev_sample_evt_df.pfp.trk.truth.p.pdg) == 2212)) &
    (abs(dev_sample_evt_df.pfp.trk.len) > 3) & (abs(dev_sample_evt_df.pfp.trackScore) > 0.5) & (dev_sample_evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for full sample
#Use the in built score
cm_df = cm_df[cm_df.pfp.is_exiting == False]
cm_df = cm_df[CutMasks.is_MIP_candidate_mask(cm_df)]
cm, x_labs, y_labs = build_pid_confusion_matrix(cm_df, bdtg_threshold = CTE.BDT_proton_max_score, key = ('pfp', 'trk', 'bdt_proton_score', '', '', ''))
cm_plot = ConfusionMatricesUtils.plot_confusion_matrix(cm, x_labs, y_labs)

save_path = os.path.join(save_fig_dir, "BDT_proton_confusion_matrix_1e20_sample.pdf")
cm_plot.savefig(save_path, bbox_inches='tight', dpi=dpi)
print(f"Saved: {save_path}")
plt.show()




'''
cut_mask = CutMasks.t0_cut_mask(evt_df) & CutMasks.nu_score_cut_mask(evt_df)  & CutMasks.track_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(evt_df,SLICE_LEVELS)
cm_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)|(abs(evt_df.pfp.trk.truth.p.pdg) == 2212)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Exiting for training
cm_df = cm_df[cm_df.pfp.is_exiting == True]
cm_df = cm_df[CutMasks.is_MIP_candidate_mask(cm_df)]
cm, x_labs, y_labs = build_pid_confusion_matrix(cm_df, bdtg_threshold = best_bdtg['cut'], key = ('pfp', 'BDTG_score_proton', '', '', '', ''))
cm_plot = ConfusionMatricesUtils.plot_confusion_matrix(cm, x_labs, y_labs)
cm_plot.savefig(save_fig_dir + "/confusion_matrix_exiting_p", dpi=300)
plt.show()


cut_mask = CutMasks.t0_cut_mask(evt_df) & CutMasks.nu_score_cut_mask(evt_df)  & CutMasks.track_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(evt_df,SLICE_LEVELS)
cm_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)|(abs(evt_df.pfp.trk.truth.p.pdg) == 2212)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]
#ALL for training
cm_df = cm_df[CutMasks.is_MIP_candidate_mask(cm_df)]
cm, x_labs, y_labs = build_pid_confusion_matrix(cm_df, bdtg_threshold = best_bdtg['cut'], key = ('pfp', 'BDTG_score_proton', '', '', '', ''))
cm_plot = ConfusionMatricesUtils.plot_confusion_matrix(cm, x_labs, y_labs)
cm_plot.savefig(save_fig_dir + "/confusion_matrix_all_p", dpi=300)
plt.show()
'''

# Training for µ/π separation

In [ ]:
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/BDTTrainingMuPi"
if not os.path.exists(save_fig_dir):
    os.makedirs(save_fig_dir)

In [ ]:


col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')
col_chi2_exp_pol  = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_chi2_exp_pol_3var = ('pfp', 'trk', 'chi2_exp_pol_3var', '', '', '')
col_frac_50 = ('pfp', 'trk', 'frac50', '', '', '')
col_scatter_angle_ratio = ('pfp', 'scatter_angle_ratio', '', '', '', '')
col_max_daughter_hits = ('pfp', 'max_daughter_hits', '', '', '', '')

cut_mask = CutMasks.t0_cut_mask(evt_df)  & CutMasks.nu_score_cut_mask(evt_df)  & CutMasks.is_inside_FV_cut_mask(evt_df) & evt_df.slc.cut.obvious_cosmic & (evt_df.truth.nu_categ != "cosmic")

BDT_columns_mupi = [
    col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_scatter_angle_ratio, col_max_daughter_hits
]


#Define the track df
signal_df = evt_df[
    cut_mask &
    (abs(evt_df.pfp.trk.truth.p.pdg) == 13) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 211) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for training
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]
signal_df = signal_df[signal_df.pfp.is_exiting == False]

#Only with good values
bdt_mask_signal = BDTTrainingUtils.bdt_quality_mask(signal_df, BDT_columns_mupi)
bdt_mask_bkg    = BDTTrainingUtils.bdt_quality_mask(bkg_df, BDT_columns_mupi)
signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]

#MIP mask
signal_df = signal_df[CutMasks.is_MIP_candidate_mask(signal_df)]
bkg_df    = bkg_df[CutMasks.is_MIP_candidate_mask(bkg_df)]

print(len(signal_df))
print(len(bkg_df))


In [ ]:
model_vec_mupi = BDTTrainingUtils.create_models_for_columns(BDT_columns_mupi)
trained_models_mupi, X_train_mupi, X_test_mupi, y_train_mupi, y_test_mupi = BDTTrainingUtils.train_all_models(
    signal_df,
    bkg_df,
    BDT_columns_mupi,
    model_vec_mupi,
    test_size=0.3
)

In [ ]:
bdt_scores_mupi_train, bdt_scores_mupi_test   = BDTTrainingUtils.get_scores(trained_models_mupi["BDT"], X_train_mupi, X_test_mupi)
bdtg_scores_mupi_train, bdtg_scores_mupi_test   = BDTTrainingUtils.get_scores(trained_models_mupi["BDTG"], X_train_mupi, X_test_mupi)
bdtb_scores_mupi_train, bdtb_scores_mupi_test   = BDTTrainingUtils.get_scores(trained_models_mupi["BDTB"], X_train_mupi, X_test_mupi)
rf_scores_mupi_train, rf_scores_mupi_test   = BDTTrainingUtils.get_scores(trained_models_mupi["RF"], X_train_mupi, X_test_mupi)
xgb_scores_mupi_train, xgb_scores_mupi_test   = BDTTrainingUtils.get_scores(trained_models_mupi["XGB"], X_train_mupi, X_test_mupi)

fig = plt.figure(figsize=(6,6))
#BDTTrainingUtils.plot_roc(y_test_mupi, bdt_scores_mupi_test, "BDT")
BDTTrainingUtils.plot_roc(y_test_mupi, bdtg_scores_mupi_test, "BDTG")
#BDTTrainingUtils.plot_roc(y_test_mupi, bdtb_scores_mupi_test, "BDTB")
#BDTTrainingUtils.plot_roc(y_test_mupi, rf_scores_mupi_test, "RF")*
BDTTrainingUtils.plot_roc(y_test_normal, bdtb_scores_test, "XGB")
plt.plot([0,1],[0,1], "k--")
plt.xlabel("Background efficiency")
plt.ylabel("Signal efficiency")
plt.legend()
plt.show()

save_path = os.path.join(save_fig_dir, "ROC_curves.pdf")
fig.savefig(save_path, format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
save_path = os.path.join(save_fig_dir, "correlation_matrices.pdf")
corr_signal, corr_bkg = BDTTrainingUtils.plot_correlation_matrices(signal_df, bkg_df, BDT_columns_mupi, name_map = BDTTrainingUtils.variables_name_map, save_folder = save_path, signal_title=r"$\mu$",
                               bkg_title=r"$\pi$")

In [ ]:

save_path = os.path.join(save_fig_dir, "Importance.pdf")
df_imp = BDTTrainingUtils.plot_importance_no_transform(trained_models_mupi["BDTG"], BDT_columns_mupi, name_map = BDTTrainingUtils.variables_name_map, save_path = save_path)
df_imp = BDTTrainingUtils.plot_importance_no_transform(trained_models_mupi["BDT"], BDT_columns_mupi, name_map = BDTTrainingUtils.variables_name_map, save_path = save_path)
#df_imp = BDTTrainingUtils.plot_importance_no_transform(trained_models_mupi["BDTB"], BDT_columns_mupi, name_map = BDTTrainingUtils.variables_name_map, save_path = save_path)



In [ ]:
sig_train = bdt_scores_mupi_train[y_train_mupi==1]
sig_test  = bdt_scores_mupi_test[y_test_mupi==1]
bkg_train = bdt_scores_mupi_train[y_train_mupi==0]
bkg_test  = bdt_scores_mupi_test[y_test_mupi==0]
save_path = os.path.join(save_fig_dir, "training_test_BDT_response.pdf")
BDTTrainingUtils.plot_response(sig_train_bdt, sig_test_bdt, bkg_train_bdt, bkg_test_bdt, title = "BDT Response", signal_label=r"$\mu$", bkg_label=r"$\pi$", save_path = save_path)

sig_train = bdtg_scores_mupi_train[y_train_mupi==1]
sig_test  = bdtg_scores_mupi_test[y_test_mupi==1]
bkg_train = bdtg_scores_mupi_train[y_train_mupi==0]
bkg_test  = bdtg_scores_mupi_test[y_test_mupi==0]
save_path = os.path.join(save_fig_dir, "training_test_BDTG_response.pdf")
BDTTrainingUtils.plot_response(sig_train, sig_test, bkg_train, bkg_test, "BDTG Response", signal_label=r"$\mu$", bkg_label=r"$\pi$", save_path = save_path)

sig_train = xgb_scores_mupi_train[y_train_mupi==1]
sig_test  = xgb_scores_mupi_test[y_test_mupi==1]
bkg_train = xgb_scores_mupi_train[y_train_mupi==0]
bkg_test  = xgb_scores_mupi_test[y_test_mupi==0]
save_path = os.path.join(save_fig_dir, "training_test_XGB_response.pdf")
BDTTrainingUtils.plot_response(sig_train, sig_test, bkg_train, bkg_test, "XBG Response", signal_label=r"$\mu$", bkg_label=r"$\pi$", save_path = save_path)


In [ ]:
with open("test_bdts/bdt_model_muon_pion.pkl", "wb") as f:
    pickle.dump(trained_models_mupi["BDT"], f)
with open("test_bdts/bdtg_model_muon_pion.pkl", "wb") as f:
    pickle.dump(trained_models_mupi["BDTG"], f)
with open("test_bdts/xgb_model_muon_pion.pkl", "wb") as f:
    pickle.dump(trained_models_mupi["XGB"], f)

In [ ]:
dev_sample_evt_df = dev_sample_matchdf

BDT_input_columns = [
    col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_scatter_angle_ratio, col_max_daughter_hits
]

col_BDT_score_muon_pion= ('pfp','BDT_score_muon_pion','','','','')

dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "test_bdts/bdt_model_muon_pion.pkl",
    BDT_input_columns,
    col_BDT_score_muon_pion
)

col_BDTG_score_muon_pion= ('pfp','BDTG_score_muon_pion','','','','')

dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "bdtg_model_muon_pion.pkl",
    BDT_input_columns,
    col_BDTG_score_muon_pion
)

col_XGB_score_muon_pion= ('pfp','XGB_score_muon_pion','','','','')

dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "test_bdts/xgb_model_muon_pion.pkl",
    BDT_input_columns,
    col_XGB_score_muon_pion
)

In [ ]:
'''
xmlWeightsFile_mp = "/exp/sbnd/data/users/lpelegri/cc1pi/CAF_analisis/UtilsCutsAndVars/BDT_muon_pion_weights.xml"
output_column = ('pfp','trk','og_bdt_muon_pion_score','','','')
dev_sample_evt_df = make_cc1pidf.add_TMVA_BDT_muon_pion_column(dev_sample_evt_df, xmlWeightsFile_mp, output_column)

xmlWeightsFile_mp = "/home/lpelegri/cafpyana/analysis_village/cc1pi/BDTs/cafpyana_muon_pion_TMVAClassification_BDT.weights.xml"
output_column = ('pfp','trk','retrain_bdt_muon_pion_score','','','')
dev_sample_evt_df = make_cc1pidf.add_TMVA_BDT_muon_pion_column(dev_sample_evt_df, xmlWeightsFile_mp, output_column)
'''

In [ ]:

cut_mask = (evt_df.slc.cut.obvious_cosmic & evt_df.slc.cut.t0 & CutMasks.is_inside_FV_cut_mask(evt_df) & (evt_df.slc.nu_score > CTE.min_nu_score) & evt_df.slc.cut.track
        & evt_df.slc.cut.MIP_candidates & evt_df.slc.cut.shower & evt_df.slc.cut.angle)
plot_df = dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & (dev_sample_evt_df.pfp.is_exiting == False)]

config_BDT_muon_pion_score = HistogramConfig(
    data_column=('pfp','BDT_score_muon_pion','','','',''),
    bins=np.linspace(-0.6, 1.5, 51),
    xlabel='BDT muon/pion score',
    ylabel='Entries',
    title='BDT muon/pion Score'
)

config_BDTG_muon_pion_score = HistogramConfig(
    data_column=('pfp','BDTG_score_muon_pion','','','',''),
    bins=np.linspace(-6, 6, 51),
    xlabel='BDTG muon/pion score',
    ylabel='Entries',
    title='BDTG muon/pion Score'
)

config_XGB_muon_pion_score = HistogramConfig(
    data_column=('pfp','XGB_score_muon_pion','','','',''),
    bins=np.linspace(0, 1, 51),
    xlabel='XGB muon/pion score',
    ylabel='Entries',
    title='XGB muon/pion Score'
)


'''
config_TMVA_og_BDT_muon_pion_score = HistogramConfig(
    data_column=('pfp','trk','og_bdt_muon_pion_score','','',''),
    bins=np.linspace(-0.25, 0.6, 51),
    xlabel='TMVA og BDT muon/pion score',
    ylabel='Entries',
    title='TMVA BDT muon/pion Score'
)

config_TMVA_retrain_BDT_muon_pion_score = HistogramConfig(
    data_column=('pfp','trk','retrain_bdt_muon_pion_score','','',''),
    bins=np.linspace(-0.25, 0.6, 51),
    xlabel='TMVA retrain BDT muon/pion score',
    ylabel='Entries',
    title='TMVA BDT muon/pion Score'
)
'''
plot_stacked_histogram(
    plot_df,
    config=config_BDT_muon_pion_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)

plot_stacked_histogram(
    plot_df,
    config=config_XGB_muon_pion_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)

plot_stacked_histogram(
    plot_df,
    config=config_BDTG_muon_pion_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)
'''
plot_stacked_histogram(
    plot_df,
    config=config_TMVA_og_BDT_muon_pion_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)

plot_stacked_histogram(
    plot_df,
    config=config_TMVA_retrain_BDT_muon_pion_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)
'''

In [ ]:
def get_muon_pion_slice_mask(df):
    """
    Returns a mask for slices that contain exactly one true muon MIP candidate 
    and exactly one true pion MIP candidate.
    """
    pdg_col = ('pfp', 'trk', 'truth', 'p', 'pdg', '')
    SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]

    # 1. Get the particle-level MIP candidate mask
    # This uses your existing function logic
    mip_particles = CutMasks.is_MIP_candidate_mask(df)

    # 2. Identify True Muons and True Pions that are ALSO MIP candidates
    is_muon_mip = (df[pdg_col].abs() == 13) & mip_particles
    is_pion_mip = (df[pdg_col].abs() == 211) & mip_particles

    # 3. Count these specific candidates per slice
    # transform('sum') broadcasts the count to all rows in the slice
    muon_mip_count = is_muon_mip.groupby(level=SLICE_LEVELS).transform('sum')
    pion_mip_count = is_pion_mip.groupby(level=SLICE_LEVELS).transform('sum')
    
    # 4. Final condition: The slice must have exactly one of each MIP-quality particle
    slice_condition = (muon_mip_count == 1) & (pion_mip_count == 1)
    
    return slice_condition

In [ ]:
def get_contained_slice_mask(df):
    """
    Returns a mask that is True for all particles in a slice ONLY if 
    EVERY particle in that slice has is_exiting == False.
    """
    exit_col = ('pfp', 'is_exiting', '', '', '', '')
    SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]

    # 1. Identify particles that are NOT exiting
    is_contained_particle = (df[exit_col] == False)

    # 2. Group by slice and check if the 'is_contained' condition is True for ALL rows
    # transform('all') broadcasts the result back to the original dataframe shape
    all_contained_mask = is_contained_particle.groupby(level=SLICE_LEVELS).transform('all')

    return all_contained_mask

In [ ]:
import matplotlib.pyplot as plt

# 1. Get the raw numbers
print("BDT")
cm_raw, labels = BDTTrainingUtils.prepare_muon_pion_cm(dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & get_muon_pion_slice_mask(dev_sample_evt_df) & (dev_sample_evt_df.truth.nu_categ == "CC1pi") & get_contained_slice_mask(dev_sample_evt_df)], score_col = ('pfp', 'BDT_score_muon_pion', '', '', '', ''))
ConfusionMatricesUtils.plot_confusion_matrix(cm_raw, labels, cmap=sunset_cmap)

print("BDTG")
cm_raw, labels = BDTTrainingUtils.prepare_muon_pion_cm(dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & get_muon_pion_slice_mask(dev_sample_evt_df) & (dev_sample_evt_df.truth.nu_categ == "CC1pi") & get_contained_slice_mask(dev_sample_evt_df)], score_col = ('pfp','BDTG_score_muon_pion', '', '', '', ''))
ConfusionMatricesUtils.plot_confusion_matrix(cm_raw, labels, cmap=sunset_cmap)


print("XGB")
cm_raw, labels = BDTTrainingUtils.prepare_muon_pion_cm(dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & get_muon_pion_slice_mask(dev_sample_evt_df) & (dev_sample_evt_df.truth.nu_categ == "CC1pi") & get_contained_slice_mask(dev_sample_evt_df)], score_col = ('pfp','XGB_score_muon_pion', '', '', '', ''))
ConfusionMatricesUtils.plot_confusion_matrix(cm_raw, labels, cmap=sunset_cmap)

'''
print("TMVA OG")
cm_raw, labels = BDTTrainingUtils.prepare_muon_pion_cm(dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & get_muon_pion_slice_mask(dev_sample_evt_df) & (dev_sample_evt_df.truth.nu_categ == "CC1pi") & get_contained_slice_mask(dev_sample_evt_df)], score_col = ('pfp','trk', 'og_bdt_muon_pion_score', '', '', ''))
ConfusionMatricesUtils.plot_confusion_matrix(cm_raw, labels, cmap=sunset_cmap)

print("TMVA retrain")
cm_raw, labels = BDTTrainingUtils.prepare_muon_pion_cm(dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & get_muon_pion_slice_mask(dev_sample_evt_df) & (dev_sample_evt_df.truth.nu_categ == "CC1pi") & get_contained_slice_mask(dev_sample_evt_df)], score_col = ('pfp','trk', 'retrain_bdt_muon_pion_score', '', '', ''))
ConfusionMatricesUtils.plot_confusion_matrix(cm_raw, labels, cmap=sunset_cmap)
'''

# Save into root

In [ ]:
import uproot
import pandas as pd
import numpy as np

# Cuts and Columns
chi2_p_cut = 80
chi2_mu_cut = 20
len_cut = 10

col_chi2_mu      = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p       = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_chi2_exp_pol = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_frac_50      = ('pfp', 'trk', 'frac50', '', '', '')
col_len          = ('pfp', 'trk', 'len', '', '', '')

BDT_columns = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_frac_50]

# Initial Cut Mask
cut_mask = CutMasks.t0_cut_mask(evt_df) & CutMasks.nu_score_cut_mask(evt_df) & (evt_df.truth.nu_categ != "cosmic")

# Define signal (Muons/Pions) and background (Protons)
signal_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
].copy()

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 2212) & (evt_df.pfp.trk.truth.p.end_process == 7) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
].copy()

# Fix: Actually apply the 'Not exiting' filter
signal_df = signal_df[signal_df.pfp.is_exiting == False]
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]

# Apply BDT quality and PID cuts
bdt_mask_signal = BDTTrainingUtils.bdt_quality_mask(signal_df, BDT_columns)
bdt_mask_bkg    = BDTTrainingUtils.bdt_quality_mask(bkg_df, BDT_columns)

signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]

signal_df = signal_df[(signal_df[col_chi2_mu] < chi2_mu_cut) & (signal_df[col_chi2_p] > chi2_p_cut) & (signal_df[col_len] > len_cut)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < chi2_mu_cut) & (bkg_df[col_chi2_p] > chi2_p_cut) & (bkg_df[col_len] > len_cut)]

# 1. Define mapping and preparation helper
column_mapping = {
    col_chi2_exp_pol: 'chi2_exp_pol0',
    col_frac_50:      'fraction_50_drift_percent',
    col_chi2_p:       'chi2p',
    col_chi2_mu:      'chi2mu'
}

def prepare_for_root_dict(df, mapping):
    """Renames columns and returns a dict of numpy arrays for uproot writing."""
    # Select only the columns needed
    subset = df[list(mapping.keys())].copy()
    # Create dictionary with new names and cast to float64 (double)
    return {mapping[k]: subset[k].values.astype(np.float64) for k in mapping}

# 2. Prepare the data
signal_dict = prepare_for_root_dict(signal_df, column_mapping)
bkg_dict    = prepare_for_root_dict(bkg_df, column_mapping)

# 3. Save to ROOT file
file_path = "/exp/sbnd/data/users/lpelegri/TransferFolder/cafpyana_proton_optimization_trees.root"
with uproot.recreate(file_path) as f:
    f["tree_muonlike"] = signal_dict
    f["tree_proton"]   = bkg_dict

print(f"ROOT file '{file_path}' created successfully.")

In [ ]:
chi2_p_cut = 80
chi2_mu_cut = 20
len_cut = 10
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')

col_chi2_exp_pol  = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_chi2_exp_pol_3var = ('pfp', 'trk', 'chi2_exp_pol_3var', '', '', '')
col_frac_50 = ('pfp', 'trk', 'frac50', '', '', '')
col_scatter_angle_ratio = ('pfp', 'scatter_angle_ratio', '', '', '', '')
col_max_daughter_hits = ('pfp', 'max_daughter_hits', '', '', '', '')
#cut_mask = CutMasks.t0_cut_mask(evt_df)  &  CutMasks.nu_score_cut_mask(evt_df)  & CutMasks.track_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(evt_df,SLICE_LEVELS)
cut_mask = CutMasks.t0_cut_mask(evt_df)  & CutMasks.nu_score_cut_mask(evt_df)  & (evt_df.truth.nu_categ != "cosmic")


#Define the track df
signal_df = evt_df[
    cut_mask &
    (abs(evt_df.pfp.trk.truth.p.pdg) == 13) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 211) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]
#Not exiting for training
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]
signal_df = signal_df[signal_df.pfp.is_exiting == False]
BDT_columns_mupi = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_scatter_angle_ratio, col_max_daughter_hits]
bdt_mask_signal = bdt_quality_mask(signal_df, BDT_columns_mupi)
bdt_mask_bkg    = bdt_quality_mask(bkg_df, BDT_columns_mupi)
signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]
signal_df = signal_df[(signal_df[col_chi2_mu] < chi2_mu_cut) & (signal_df[col_chi2_p] > chi2_p_cut) & (signal_df[col_len] > len_cut)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < chi2_mu_cut) & (bkg_df[col_chi2_p] > chi2_p_cut) & (bkg_df[col_len] > len_cut)]

# 1. Define mapping and preparation helper
column_mapping = {
    col_frac_50:      'fraction_50_drift_percent',
    col_chi2_p:       'chi2p',
    col_chi2_mu:      'chi2mu',
    col_scatter_angle_ratio: 'track_mcs_scatter_max_ratio',
    col_max_daughter_hits: 'max_daughter_hits'
}
print(signal_df[signal_df.pfp.max_daughter_hits != 0].pfp.max_daughter_hits)
def prepare_for_root_dict(df, mapping):
    """Renames columns and returns a dict of numpy arrays for uproot writing."""
    # Select only the columns needed
    subset = df[list(mapping.keys())].copy()
    # Create dictionary with new names and cast to float64 (double)
    return {mapping[k]: subset[k].values.astype(np.float64) for k in mapping}

# 2. Prepare the data
signal_dict = prepare_for_root_dict(signal_df, column_mapping)
bkg_dict    = prepare_for_root_dict(bkg_df, column_mapping)

# 3. Save to ROOT file
file_path = "/exp/sbnd/data/users/lpelegri/TransferFolder/cafpyana_muon_pion_optimization_trees.root"
with uproot.recreate(file_path) as f:
    f["tree_muon"] = signal_dict
    f["tree_pion"]   = bkg_dict

print(f"ROOT file '{file_path}' created successfully.")

In [ ]:
print(signal_df[signal_df.pfp.max_daughter_hits != 0].pfp.max_daughter_hits)